# Clustering Impact Comparison (TF-IDF based)
This notebook compares the search results of the Information Retrieval System with and without **TF-IDF based** cluster re-ranking.

In [3]:
import requests
import time
import pandas as pd

SEARCH_API_URL = "http://127.0.0.1:8003/search/"
# IMPORTANT: Make sure you have run the clustering process with 'use_bert': false for this dataset
DATASET_NAME = "antique" # or 'beir/quora', 'lotte/lifestyle/dev'
QUERY = "what is the capital of france" # Example query

## 1. Search without Clustering (Baseline)

In [4]:
payload_no_clustering = {
    "dataset_name": DATASET_NAME,
    "query": QUERY,
    "model_type": "hybrid",
    "enable_cluster_reranking": False
}

print("Running baseline search...")
start_time = time.time()
response_no_clustering = requests.post(SEARCH_API_URL, json=payload_no_clustering)
end_time = time.time()

time_no_clustering = end_time - start_time
results_no_clustering = response_no_clustering.json()["results"]

df_no_clustering = pd.DataFrame(results_no_clustering)

print(f"Search time without clustering: {time_no_clustering:.4f} seconds")
print("Results without clustering:")
display(df_no_clustering)

Running baseline search...
Search time without clustering: 25.4587 seconds
Results without clustering:


,doc_id,score,cluster_id
0,1379520_2,0.800000,2
1,3282965_0,0.432492,2
2,2008308_4,0.299491,2
3,1182771_7,0.286080,0
4,4034012_0,0.263640,2
5,1730696_2,0.200000,2
6,574181_3,0.175382,2
7,1649713_3,0.133648,2
8,1500264_2,0.117795,2
9,3725220_1,0.102118,2


## 2. Search with TF-IDF Clustering

In [5]:
payload_with_clustering = {
    "dataset_name": DATASET_NAME,
    "query": QUERY,
    "model_type": "hybrid",
    "enable_cluster_reranking": True
}

print("Running search with TF-IDF cluster re-ranking...")
start_time = time.time()
response_with_clustering = requests.post(SEARCH_API_URL, json=payload_with_clustering)
end_time = time.time()

time_with_clustering = end_time - start_time
results_with_clustering = response_with_clustering.json()["results"]

df_with_clustering = pd.DataFrame(results_with_clustering)

print(f"Search time with TF-IDF clustering: {time_with_clustering:.4f} seconds")
print("Results with TF-IDF clustering:")
display(df_with_clustering)

Running search with TF-IDF cluster re-ranking...
Search time with TF-IDF clustering: 0.3052 seconds
Results with TF-IDF clustering:


,doc_id,score,cluster_id
0,1379520_2,1.800000,2
1,3282965_0,1.432492,2
2,2008308_4,1.299491,2
3,4034012_0,1.263640,2
4,1730696_2,1.200000,2
5,574181_3,1.175382,2
6,1649713_3,1.133648,2
7,1500264_2,1.117795,2
8,3725220_1,1.102118,2
9,1182771_7,0.286080,0


## 3. Comparison

In [6]:
print(f"Time difference (with - without): {time_with_clustering - time_no_clustering:.4f} seconds")

print("Documents in top 10 without clustering:", set(df_no_clustering["doc_id"]))
print("Documents in top 10 with TF-IDF clustering:", set(df_with_clustering["doc_id"]))

common_docs = set(df_no_clustering["doc_id"]).intersection(set(df_with_clustering["doc_id"]))
print(f"Number of common documents in top 10: {len(common_docs)}")

Time difference (with - without): -25.1536 seconds
Documents in top 10 without clustering: {'4034012_0', '2008308_4', '3725220_1', '1649713_3', '1730696_2', '574181_3', '3282965_0', '1500264_2', '1182771_7', '1379520_2'}
Documents in top 10 with TF-IDF clustering: {'4034012_0', '2008308_4', '3725220_1', '1649713_3', '574181_3', '3282965_0', '1500264_2', '1730696_2', '1182771_7', '1379520_2'}
Number of common documents in top 10: 10
